In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-40s :: %(message)s'
)


## Inverse obstacle problem with Dirichlet boundary condition

We consider the operator that maps the shape of a sound-soft obstacle $D$ to the far-field measurements. 
The scattering problem is described by

$$
        \begin{cases}
            \Delta u +\kappa^2 u = 0 & \text{ in } \mathbb{R}^2\backslash\overline{D}\\
             u = 0  & \text{ on } \partial D\\
            \displaystyle{\lim_{r\to\infty}}r^{\frac{1}{2}}(\frac{\partial u^s}{\partial r}-i\kappa u^s)=0 & \text{ for } r=|x|,
        \end{cases}
$$
where $u=u^s+u^i$ is the total field generated by a plane incident wave $u^i(x)=\exp(i\kappa x\cdot d)$ in direction 
$d\in S^1$, 
and $D$ is a bounded obstacle in $\mathbb{R}^2$ with $\partial D\in\mathcal{C}^2$.
The far field pattern $u^{\infty}$ of the scattered field $u^s$ is defined by the asymptotic relation
$$
u^s(x) = \frac{e^{i\kappa|x|}}{\sqrt{|x}} u^{\infty}\left(\frac{x}{|x|}\right)\left(1+O\left(\frac{1}{|x|}\right)\right),\qquad 
|x|\to \infty.
$$ 
The forward operator maps a parameterization of $\partial D$ to a complex matrix, the columns of which are far field patterns 
corresponding to incident waves from different directions.      

Optionally, we can manually choose the type of parameterization of the curve.

In [ ]:
from regpy.vecsps.curve import GenTrigSpc,StarTrigRadialFcts
domain = StarTrigRadialFcts(dim=64)
#domain = GenTrigSpc(n_sample=64)

In [ ]:
"""from  dirichlet_op import DirichletOp
op = DirichletOp(
    kappa = 4,
    inc_waves=4,
    meas =64,
    R_meas = np.inf,
    N_ieq =64,
    domain=domain
)"""

### Neumann boundary condition
Instead of the Dirichlet condition $u=0$ on $\partial D$, we can also consider a Neumann condition for the total field, 
corresponding to sound-hard obstacles:
$$
\frac{\partial u}{\partial \nu}=0\qquad \text{on }\partial D. 
$$

In [ ]:
from neumann_op import NeumannOp
op = NeumannOp(
    kappa = 4,
    R_inc = np.inf,
    inc_waves=4,
    meas =64,
    R_meas=np.inf,
    N_ieq=64,
    domain=domain
)

### Transmission conditions 
Scattering by a homogeneous penetrable obstacle can be described by transmission conditions: 
$$
        \begin{cases}
            \Delta u^{int} +\kappa_i^2 u^{int} = 0 & \text{ in } D \\
            \Delta u^{s} +\kappa_e^2 u^{s} = 0 & \text{ in } \mathbb{R}^2\backslash\overline{D}\\
             u^{int}=u & \text{ on } \partial D\\
            \frac{\partial u^{int}}{\partial\nu}=\rho\frac{\partial u}{\partial\nu} & \text{ on }\partial D\\
            \displaystyle{\lim_{r\to\infty}}r^{\frac{1}{2}}(\frac{\partial u^s}{\partial r}-i\kappa u^s)=0 &\text{ for } r=|x|.
        \end{cases}
$$
Here $\rho\in\mathbb{C}\backslash 0$, and $u=u^{s}+u^{i}$ is the total field in $\mathbb{R}^2\backslash\overline{D}$.


In [ ]:
"""from transmission_op import TransmissionOp
op = TransmissionOp(
    kappa_in = 4,
    kappa_ex = 8,
    R_inc = np.inf,
    inc_waves=1,
    rho = 2., 
    R_meas =5.,
    meas =64,
    N_ieq=64,
    domain=domain
)"""

import some curve as ground truth and create corresponding synthetic data

In [ ]:
from regpy.vecsps.curve import Apple,Kite, Nonsym_shape,Three_lobes,Peanut,Round_rect,Smoothed_rectangle, Pinched_ellipse
exact_data, exact_solution = op.create_synthetic_data(Apple,N_ieq_synth=2*op.N_ieq)


plot the exact synthetic data

In [ ]:
x= op.codomain.coords[0]
fig, axs = plt.subplots(1, 2,figsize=(10,5))
number2show = np.min([4,op.N_inc])
for j in range(number2show):
    y = exact_data[:,j]
    axs[0].plot(x,np.abs(y))
    axs[1].plot(x,np.unwrap(np.angle(y)))
fig.suptitle(f'farfield patterns for first {number2show} of {exact_data.shape[1]} incident waves')
axs[0].set_title('absolute values')
axs[1].set_title('phases')
axs[0].set_xlabel('Measurement direction (rad)')
_= axs[1].set_xlabel('Measurement direction (rad)')


Get initial guess, add noise to data, and create setting

In [ ]:
from regpy.solvers import Setting
from regpy.hilbert import L2, Sobolev

#Initial guess
init = op.domain.circle(radius = 0.45)

setting = Setting(op=op, 
                  penalty=Sobolev(index=1.6), 
                  data_fid=L2,
                  exact_data=exact_data
                  )

# add Gaussian noise to exact data
relative_noise_level = 0.02
setting.add_Gaussian_noise(relative_noise_level=0.01)

perform the inversion

In [ ]:
from regpy.solvers.nonlinear.irgnm import IrgnmCG
from regpy.solvers.nonlinear.newton import NewtonCG
import regpy.stoprules as rules

#Solver: NewtonCG or IrgnmCG

solver = NewtonCG(
    setting, 
    init = init.coeff,
    cgmaxit=100, 
    rho=0.6
)
"""

solver = IrgnmCG(
    setting, data,
    regpar=1.,
    regpar_step=0.5,
    init=init
)
"""
stoprule = (
    rules.CountIterations(25) +
    rules.Discrepancy(
        relative_noise_level,
        setting = setting,
        relative_noise_level = True,
        tau=2.1
    )
)
import cProfile, pstats

reco, reco_data = solver.run(stoprule)


Plot results

In [ ]:
def add_scalebar(
    ax,
    length,
    label,
    location=(0.1, 0.1),
    linewidth=3,
    text_offset=-0.05
):
    """
    Add a horizontal scale bar to an axes.

    location is in axes coordinates (0–1)
    """
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x0 = xlim[0] + location[0] * (xlim[1] - xlim[0])
    y0 = ylim[0] + location[1] * (ylim[1] - ylim[0])

    ax.plot([x0, x0 + length], [y0, y0], lw=linewidth,    color='black')
    ax.text(
        x0 + length / 2,
        y0 - text_offset * (ylim[1] - ylim[0]),
        label,
        ha="center",
        va="top"
    )


In [ ]:
fig, axs = plt.subplots(1, 2,figsize=(10,5))
axs[0].set_title('obstacle')
axs[1].set_title('farfield (real part,first inc. wave)')

exact_z = exact_solution.z
reco_z = op.domain.coeff2curve(reco).z
axs[0].plot(*(np.concatenate((exact_z,exact_z[:,:1]),axis=1)),label='exact')
axs[0].plot(*(np.concatenate((reco_z, reco_z[:,:1]), axis=1)),label='reco')
axs[0].plot(*(np.concatenate((init.z, init.z[:,:1]), axis=1)),label='init')
axs[0].legend()
kappa =op.kappa if hasattr(op,'kappa') else op.kappa_ex
add_scalebar(axs[0],length=np.pi/kappa.real,label='$\lambda/2$',location=(0,0))

axs[1].plot(op.codomain.coords[0][:,0], exact_data.imag[:,0], label='exact')
axs[1].plot(op.codomain.coords[0][:,0], reco_data.real[:,0], label='reco')
axs[1].plot(op.codomain.coords[0][:,0], setting.data.real[:,0], '.', label='measured')
axs[1].legend()
plt.show()